# AuraScribble Auto-Retrain — Kaggle Edition (V8)

Triggered automatically by GitHub Actions when ≥500 new Firebase corrections accumulate.

**Flow:**
1. Read Firebase service account from Kaggle Secrets
2. Read base bundle (code + datasets) from attached Kaggle Dataset
3. Download latest checkpoint from Firebase Storage (`models/checkpoint_best.pt`)
4. Download all new corrections from `training_data/new/`
5. Build manifests + train V8 (CTC hybrid, 20 epochs)
6. Upload new ONNX + checkpoint to Firebase
7. Archive processed corrections to `training_data/processed/`

**No human interaction required.** The notebook is idempotent and safe to re-run.

In [ ]:
import os, sys, json, shutil
from pathlib import Path

# === Kaggle paths ===
BUNDLE_BASE = Path('/kaggle/input/aurascribble-bundle')   # attached dataset
WORK_DIR    = '/kaggle/working/handwriting-model'
OUT_DIR     = '/kaggle/working/handwriting-model/output'

# The user's zip might have packed the contents under a top-level
# `colab_bundle/` folder. Auto-detect either layout.
if (BUNDLE_BASE / 'src').is_dir():
    BUNDLE_DIR = BUNDLE_BASE
elif (BUNDLE_BASE / 'colab_bundle' / 'src').is_dir():
    BUNDLE_DIR = BUNDLE_BASE / 'colab_bundle'
else:
    # Last-ditch: pick the first child dir that has src/
    candidates = [d for d in BUNDLE_BASE.iterdir() if d.is_dir() and (d / 'src').is_dir()]
    if not candidates:
        raise RuntimeError(
            f'Could not find src/ under {BUNDLE_BASE}. Contents: '
            f'{list(BUNDLE_BASE.iterdir())[:20]}'
        )
    BUNDLE_DIR = candidates[0]
print(f'Bundle root: {BUNDLE_DIR}')

# Mirror the bundle into a writable working directory
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

# Copy code + processed data (~2.2GB but fast on Kaggle SSD)
for sub in ('src', 'configs', 'scripts', 'data', 'requirements.txt'):
    s = BUNDLE_DIR / sub
    d = Path(WORK_DIR) / sub
    if s.is_dir():
        shutil.copytree(s, d)
    elif s.is_file():
        shutil.copy2(s, d)

os.chdir(WORK_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
print('cwd =', os.getcwd())
print('contents:', sorted(os.listdir('.')))
print('src/ has', len(os.listdir('src')), 'files,', 'configs/ has', len(os.listdir('configs')), 'files')

In [ ]:
# Install deps + Hebrew fonts (Kaggle already has torch, just need a few extras)
!pip -q install firebase-admin google-cloud-storage google-auth PyYAML
!apt-get -qq update > /dev/null 2>&1
!apt-get -qq install -y fonts-noto-core fonts-dejavu fonts-dejavu-core > /dev/null 2>&1
!fc-list :lang=he | head -3

In [ ]:
# Firebase service account injected by GitHub Actions before kernel push.
# (Kaggle Secrets can't be attached via the API, so we inline the JSON as base64
# — the kernel is private, only the owner can see it.)
import base64

SA_B64 = "__FIREBASE_SA_B64__"   # replaced by trigger_kaggle_retrain.py
if SA_B64.startswith('__'):
    # fallback for manual runs: try the Kaggle Secrets API
    from kaggle_secrets import UserSecretsClient
    sa_json = UserSecretsClient().get_secret('FIREBASE_SERVICE_ACCOUNT_JSON')
else:
    sa_json = base64.b64decode(SA_B64).decode('utf-8')

cred_path = Path('configs/firebase_service_account.json')
cred_path.parent.mkdir(parents=True, exist_ok=True)
cred_path.write_text(sa_json, encoding='utf-8')
json.loads(sa_json)   # validate
print('✓ Firebase service account loaded')

In [ ]:
# Download previous best checkpoint from Firebase Storage so we can resume
import firebase_admin
from firebase_admin import credentials, storage

if not firebase_admin._apps:
    firebase_admin.initialize_app(
        credentials.Certificate(str(cred_path)),
        {'storageBucket': 'aurascribblr.firebasestorage.app'},
    )
bucket = storage.bucket()

Path('models').mkdir(exist_ok=True)
blob = bucket.blob('models/checkpoint_best.pt')
RESUME = False
if blob.exists():
    blob.download_to_filename('models/checkpoint_best.pt')
    sz = os.path.getsize('models/checkpoint_best.pt') / 1024 / 1024
    print(f'✓ Downloaded base checkpoint ({sz:.1f} MB) — will resume from it')
    RESUME = True
else:
    print('⚠ No remote checkpoint yet — training will start from scratch')

In [ ]:
# Download new user corrections from Firebase Storage
import sys
sys.path.insert(0, 'src')
from download_firebase_corrections import download_corrections

n_corrections = download_corrections(
    local_dir='data/raw/training_data/new',
    credentials_path=str(cred_path),
    include_processed=False,
)
print(f'Downloaded {n_corrections} new correction JSON files')

In [ ]:
# Rebuild all_real.jsonl with corrections folded in
from prepare_raw import prepare_raw

n = prepare_raw(
    raw_dir='data/raw',
    output='data/processed/all_real.jsonl',
    append_processed=False,
)
print(f'all_real.jsonl now has {n} samples')

In [ ]:
# Generate synthetic Hebrew + mixed (same as V8)
!python src/generate_synthetic_hebrew.py \
  --output data/raw/synthetic_hebrew/hebrew_synthetic.jsonl \
  --sentences configs/hebrew_sentences_full.txt \
  --variants 40

!python src/generate_synthetic_mixed.py \
  --output data/raw/synthetic_mixed/mixed_synthetic.jsonl \
  --per-template 50

In [ ]:
# Merge real + synthetic into data/processed/all.jsonl
out = Path('data/processed/all.jsonl')
out.parent.mkdir(parents=True, exist_ok=True)
sources = [
    'data/processed/all_real.jsonl',
    'data/raw/synthetic_hebrew/hebrew_synthetic.jsonl',
    'data/raw/synthetic_mixed/mixed_synthetic.jsonl',
]
seen = set()
n_total = 0
with out.open('w', encoding='utf-8') as fout:
    for src in sources:
        sp = Path(src)
        if not sp.exists():
            continue
        for line in sp.open(encoding='utf-8'):
            obj = json.loads(line)
            first = tuple(obj['points'][0]) if obj.get('points') else ()
            key = f"{obj.get('mode')}|{obj.get('text')}|{len(obj.get('points', []))}|{first}"
            if key in seen:
                continue
            seen.add(key)
            fout.write(line if line.endswith('\n') else line + '\n')
            n_total += 1
print(f'all.jsonl: {n_total} samples')

In [ ]:
# Stratified split + curriculum
!python -u src/split_manifest.py \
  --source data/processed/all.jsonl \
  --train-out data/processed/train.jsonl \
  --val-out data/processed/val.jsonl \
  --val-ratio 0.1 \
  --seed 42 \
  --balance-modes \
  --per-mode-target 6000

!python -u src/build_curriculum_manifest.py \
  --train data/processed/train.jsonl \
  --short-out data/processed/train_short.jsonl \
  --medium-out data/processed/train_medium.jsonl \
  --iam-out data/processed/train_iam_long.jsonl \
  --max-chars-short 32 \
  --medium-min-chars 33 \
  --medium-max-chars 72 \
  --rewrite-train

In [ ]:
# Build the V8 training config (hybrid CTC+AR, full train.jsonl, resume if checkpoint exists)
import yaml, copy

with open('configs/train.yaml', 'r', encoding='utf-8') as f:
    base = yaml.safe_load(f)

cfg = copy.deepcopy(base)
cfg['model_type'] = 'hybrid'
cfg['ctc_loss_weight'] = 0.7
cfg['ar_loss_weight']  = 0.3
cfg['learning_rate'] = 3e-5
cfg['training']['learning_rate'] = 3e-5
cfg['train_manifest']        = 'data/processed/train.jsonl'
cfg['data']['train_manifest'] = 'data/processed/train.jsonl'
cfg['resume_from_checkpoint'] = bool(RESUME)
cfg['model_path']             = 'models/checkpoint_best.pt'
cfg['output_dir']             = 'output'
cfg['epochs']                 = 20
cfg['training']['epochs']     = 20
cfg['correction_loss_weight'] = 5.0

with open('configs/train_kaggle_v8.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f, allow_unicode=True)
print('Wrote configs/train_kaggle_v8.yaml')
print('  resume:', cfg['resume_from_checkpoint'])
print('  epochs:', cfg['epochs'])

In [ ]:
# Train
!python -u src/train.py --config configs/train_kaggle_v8.yaml

In [ ]:
# Predict on full val set + collapse check
!python -u src/predict.py \
  --config configs/train_kaggle_v8.yaml \
  --checkpoint output/checkpoint_best.pt \
  --manifest data/processed/val.jsonl \
  --output output/predictions.jsonl

from collections import Counter
preds = {int(json.loads(l)['id']): json.loads(l).get('prediction', '')
         for l in Path('output/predictions.jsonl').open(encoding='utf-8')}
n_total = len(preds)
n_empty = sum(1 for p in preds.values() if not p)
nonempty_unique = len({p for p in preds.values() if p})
diversity = nonempty_unique / max(1, n_total - n_empty)

print(f'total: {n_total}  empty: {n_empty} ({100*n_empty/n_total:.1f}%)')
print(f'nonempty diversity: {diversity:.2f}')

GO = (diversity >= 0.3) and (n_empty < 0.5 * n_total)
print(('✅ Safe to export' if GO else '🚨 NOT SAFE — collapse detected, abort upload'))
if not GO:
    raise RuntimeError('Quality gate failed — refusing to upload broken model')

In [ ]:
# Eval CER
!python -u src/evaluate.py \
  --manifest data/processed/val.jsonl \
  --predictions output/predictions.jsonl \
  --report output/eval_report.json

report = json.loads(Path('output/eval_report.json').read_text(encoding='utf-8'))
print(f"CER mean: {report.get('cer_mean'):.3f}")
print(f"by mode: {report.get('cer_by_mode')}")

In [ ]:
# Export ONNX
!python -u src/export_onnx.py \
  --config configs/train_kaggle_v8.yaml \
  --checkpoint output/checkpoint_best.pt \
  --trace-time 128 --trace-tokens 128 --smoke-time 128 \
  --summary output/export_summary.json

s = json.loads(Path('output/export_summary.json').read_text(encoding='utf-8'))
assert s.get('status') == 'success', f'Export failed: {s}'
print('✓ ONNX export successful')

In [ ]:
# Upload ONNX + checkpoint + vocab to Firebase Storage
uploads = [
    ('output/model.onnx',                   'models/latest_handwriting.onnx'),
    ('output/checkpoint_best.pt',           'models/checkpoint_best.pt'),
    ('output/vocab.from_checkpoint.txt',    'models/latest_vocab.txt'),
]
for local, remote in uploads:
    if os.path.exists(local):
        bucket.blob(remote).upload_from_filename(local)
        print(f'✓ uploaded {local} → gs://{bucket.name}/{remote}')
    else:
        print(f'⚠ missing local file: {local}')

In [ ]:
# Archive processed corrections (move new → processed) so we don't re-train on them
n_moved = 0
for blob in bucket.list_blobs(prefix='training_data/new/'):
    if blob.name.endswith('/') or not blob.name.endswith('.json'):
        continue
    new_path = blob.name.replace('training_data/new/', 'training_data/processed/')
    bucket.rename_blob(blob, new_path)
    n_moved += 1
print(f'Archived {n_moved} correction files to training_data/processed/')

In [ ]:
# Write a result summary so GitHub Actions can confirm success
result = {
    'status': 'success',
    'cer_mean': report.get('cer_mean'),
    'cer_by_mode': report.get('cer_by_mode'),
    'samples': report.get('samples'),
    'corrections_used': n_corrections,
    'archived': n_moved,
}
Path('/kaggle/working/result.json').write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2, ensure_ascii=False))